In [1]:
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

import sys
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import scipy
from itertools import combinations

In [2]:
sys.path.insert(0, "src")

from grn import grn

np.random.seed(5)


In [3]:
all_results = []

for network_seed in range(10):
    np.random.seed(network_seed)

    #manually adjust GRN hyperparameters in an attempt to build GRN with more non-additive effects
    G = grn().add_structure(
        n=256,
        k=1,
        alpha=1e-99,
        beta=0.75,       # 1 - 1/r
        gamma=0.25,      # 1/r
        kappa=10,        # --w
        delta_in=100,
        delta_out=1,
    )

    G.simulate_rna(
        s=1e-4,
        dt=1e-2,
        tmax=20000,
        burnin=5000,
        tol=1e-3,
        save=True,
    )

    x_wt = np.zeros(256)
    y_wt = G.rna

    single_ko_expression = np.zeros((G.n, G.n))
    single_ko_logfc = np.zeros((G.n, G.n))
    single_ko_convergence = {}

    for gene in tqdm(range(G.n), desc=f"Network {network_seed}: single-gene KOs"):

        result = G.ko_nodes(
            genes=[gene],
            stats=("new_rna", "logfc", "convergence"),
            s=1e-4,
            dt=1e-2,
            tmax=20000,
            burnin=5000,
            tol=1e-3,
        )

        single_ko_expression[gene, :] = result["new_rna"].flatten()
        single_ko_logfc[gene, :] = result["logfc"].flatten()
        single_ko_convergence[gene] = result["convergence"]

    x_single_ko = np.eye(G.n)
    y_single_ko = single_ko_expression

    pairs = list(combinations(range(G.n), 2))

    def run_double_ko(pair):
        a, b = pair

        result = G.ko_nodes(
            genes=[a, b],
            stats=("new_rna", "logfc", "convergence"),
            s=0,
            dt=1e-2,
            tmax=20000,
            burnin=5000,
            tol=1e-3,
        )

        return (
            np.asarray(result["new_rna"]).flatten(),
            np.asarray(result["logfc"]).flatten(),
            np.asarray(result["convergence"]),
        )


    results = Parallel(
        n_jobs=64,
        pre_dispatch="n_jobs",
        prefer="processes",
        verbose=10,
    )(
        delayed(run_double_ko)(pair)
        for pair in pairs
    )

    double_expression = np.asarray(
        [r[0] for r in results],
        dtype=np.float32
    )

    double_logfc = np.asarray(
        [r[1] for r in results],
        dtype=np.float32
    )

    double_convergence = np.asarray(
        [r[2] for r in results],
        dtype=np.float32
    )


    #now prepare as training data

    pairs = np.array(
        pairs,
        dtype=np.int32
    )

    x_double_ko = np.zeros((pairs.shape[0], G.n))

    #now fill x for doubleKO with 1s

    rows = np.arange(pairs.shape[0])
    x_double_ko[rows, pairs[:, 0]] = 1.0
    x_double_ko[rows, pairs[:, 1]] = 1.0

    y_double_ko = double_expression


    # ============================================================
    # 10 TRAINING RUNS ON THIS GRN
    # ============================================================

    for training_seed in range(10):

        import torch
        from torch import nn

        torch.manual_seed(training_seed)
        device = torch.device("cpu")

        #for more precision
        #torch.backends.cuda.matmul.allow_tf32 = False
        #torch.backends.cudnn.allow_tf32 = False


        #model implementation without dropout
        model = nn.Sequential(
            nn.Linear(256, 512),
            nn.GELU(),
            nn.Linear(512, 512),
            nn.GELU(),
            nn.Linear(512, 256)
        ).to(device)

        linear = nn.Linear(256, 256).to(device)

        #implement AdamW
        import torch.optim as optim

        optimizer = optim.AdamW(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-2
        )


        #process data, standardise, put them on GPU

        rng = np.random.default_rng(training_seed)

        n_pairs = pairs.shape[0]
        nt, nv = int(0.5 * n_pairs), int(0.1 * n_pairs)

        indices = rng.permutation(n_pairs)
        test_idx = indices[:nt]
        val_idx = indices[nt:nt + nv]
        train_idx = indices[nt + nv:]

        xtr = np.concatenate(
            [x_wt[None, :], x_single_ko, x_double_ko[train_idx]],
            axis=0
        )

        ytr = np.concatenate(
            [y_wt[None, :], y_single_ko, y_double_ko[train_idx]],
            axis=0
        )

        xv = x_double_ko[val_idx]
        yv = y_double_ko[val_idx]

        x_test = x_double_ko[test_idx]
        y_test = y_double_ko[test_idx]

        mean, scale = (
            np.mean(ytr, axis=0),
            np.maximum(ytr.std(axis=0), 1e-6)
        )

        s_ytr = (ytr - mean) / scale

        xtr_t = torch.as_tensor(
            xtr,
            dtype=torch.float32,
            device=device
        )

        s_ytr_t = torch.as_tensor(
            s_ytr,
            dtype=torch.float32,
            device=device
        )

        xv_t = torch.as_tensor(
            xv,
            dtype=torch.float32,
            device=device
        )

        yv_t = torch.as_tensor(
            yv,
            dtype=torch.float32,
            device=device
        )

        x_test_t = torch.as_tensor(
            x_test,
            dtype=torch.float32,
            device=device
        )

        y_test_t = torch.as_tensor(
            y_test,
            dtype=torch.float32,
            device=device
        )

        mean_t = torch.as_tensor(
            mean,
            dtype=torch.float32,
            device=device
        )

        scale_t = torch.as_tensor(
            scale,
            dtype=torch.float32,
            device=device
        )


        import time
        import copy

        max_epochs = 3000
        batch_size = 256
        lr = 1e-3
        patience = 40
        min_lr = 1e-5
        lr_patience = 12

        best, best_epoch, stale, lr_stale = np.inf, 0, 0, 0
        history = []
        started = time.perf_counter()
        best_model_weights = None


        for epoch in range(1, max_epochs + 1):

            model.train()

            permutation = torch.randperm(
                xtr_t.shape[0],
                device=device
            )

            total = torch.zeros((), device=device)

            for start in range(
                0,
                xtr_t.shape[0],
                batch_size
            ):

                ii = permutation[start:(start + batch_size)]

                optimizer.zero_grad(set_to_none=True)

                loss = torch.mean(
                    (model(xtr_t[ii]) - s_ytr_t[ii]) ** 2
                )

                loss.backward()
                optimizer.step()

                total += loss.detach() * len(ii)

            train_loss = float(
                total.item() / xtr_t.shape[0]
            )

            model.eval()

            with torch.inference_mode():
                val_scaled = torch.cat(
                    [
                        model(chunk)
                        for chunk in xv_t.split(1024)
                    ]
                ).cpu().numpy()

            val_pred = (
                val_scaled.astype(np.float32)
                * scale
                + mean
            )

            raw_mse = float(
                np.mean((val_pred - yv) ** 2)
            )

            if (
                not np.isfinite(train_loss)
                or not np.isfinite(raw_mse)
            ):
                raise FloatingPointError(
                    f"Non-finite loss at epoch {epoch}"
                )

            history.append({
                "epoch": epoch,
                "train_scaled_mse": train_loss,
                "val_scaled_mse": float(
                    np.mean(
                        ((val_pred - yv) / scale) ** 2
                    )
                ),
                "val_raw_mse": raw_mse,
                "learning_rate": lr,
                "elapsed_seconds":
                    time.perf_counter() - started
            })

            if raw_mse < best:

                best = raw_mse
                best_epoch = epoch
                stale = 0
                lr_stale = 0

                best_model_weights = copy.deepcopy(
                    model.state_dict()
                )

            else:

                stale += 1
                lr_stale += 1

            if epoch == 1 or epoch % 10 == 0:

                print(
                    f"network {network_seed}, "
                    f"training {training_seed}, "
                    f"epoch {epoch}: "
                    f"train scaled MSE {train_loss:.6g}; "
                    f"val RMSE {raw_mse**.5:.7g}; "
                    f"lr {lr:g}; "
                    f"{time.perf_counter()-started:.0f}s",
                    flush=True
                )

            if (
                lr_stale >= lr_patience
                and lr > min_lr
            ):

                lr = max(
                    min_lr,
                    lr * .5
                )

                for group in optimizer.param_groups:
                    group["lr"] = lr

                lr_stale = 0

            if stale >= patience:
                break


        if best_model_weights is not None:
            model.load_state_dict(
                best_model_weights
            )


        from sklearn.linear_model import LinearRegression

        # 1. Initialize standard Linear Regression (no alpha/penalty)
        linear_model = LinearRegression(
            fit_intercept=True
        )

        # 2. Fit directly on the NumPy arrays
        linear_model.fit(
            xtr.astype(np.float64),
            s_ytr.astype(np.float64)
        )

        # 3. Predict and reverse normalization
        val_scaled = linear_model.predict(
            xv.astype(np.float64)
        )

        val_pred = (
            val_scaled.astype(np.float32)
            * scale
            + mean
        )

        # 4. Calculate RMSE
        raw_mse = float(
            np.mean(
                (val_pred - yv) ** 2
            )
        )

        print(
            f"Standard Linear Baseline Validation RMSE: "
            f"{raw_mse**.5:.7g}"
        )


        # 1. Generate predictions for the Test set
        model.eval()

        with torch.inference_mode():

            mlp_test_scaled = torch.cat(
                [
                    model(chunk)
                    for chunk
                    in x_test_t.split(1024)
                ]
            ).cpu().numpy()

        mlp_pred = (
            mlp_test_scaled.astype(np.float32)
            * scale
            + mean
        )


        # Retrieve linear model predictions
        linear_test_scaled = linear_model.predict(
            x_test.astype(np.float64)
        )

        linear_pred = (
            linear_test_scaled.astype(np.float32)
            * scale
            + mean
        )


        # 2. Reconstruct test pairs and baseline predictions
        test_pairs = pairs[test_idx]

        # The additive baseline is Single A + Single B - WT
        additive = (
            y_single_ko[test_pairs[:, 0]]
            + y_single_ko[test_pairs[:, 1]]
            - y_wt
        )

        y_true = y_test

        true_r = y_true - additive

        add_pair_mse = np.mean(
            true_r ** 2,
            axis=1
        )

        add_mse = float(
            np.mean(add_pair_mse)
        )


        # Helper function for Pearson correlation
        def correlation(a, b):

            a = np.asarray(a).ravel()
            b = np.asarray(b).ravel()

            if (
                np.std(a) < 1e-15
                or np.std(b) < 1e-15
            ):
                return None

            return float(
                np.corrcoef(a, b)[0, 1]
            )


        predictions = {
            "Additive Baseline": additive,
            "Linear Regression": linear_pred,
            "MLP": mlp_pred
        }

        metrics = []
        pair_losses = {}


        # 3. Calculate core metrics
        for name, pred in predictions.items():

            e2 = (pred - y_true) ** 2

            pair_losses[name] = e2.mean(axis=1)

            implied_residual = pred - additive

            mse = float(
                np.mean(e2)
            )

            pearson_corr = correlation(
                implied_residual,
                true_r
            )

            metrics.append({
                "Model": name,
                "MSE": mse,
                "RMSE": mse ** 0.5,
                "MSE Improvement vs Additive (%)":
                    100 * (1 - mse / add_mse),
                "Pearson (Implied vs True Residual)":
                    pearson_corr if pearson_corr else 0.0,
                "Fraction Pairs < Additive MSE":
                    float(
                        np.mean(
                            pair_losses[name]
                            < add_pair_mse
                        )
                    )
            })


        df = pd.DataFrame(
            metrics
        ).set_index("Model")


        # 4. Bootstrap 95% Confidence Intervals
        # for MSE Improvement (2000 resamples)

        rng = np.random.default_rng(9001)

        boot = {
            "Linear Regression": [],
            "MLP": []
        }

        for _ in range(2000):

            ii = rng.integers(
                0,
                len(test_pairs),
                size=len(test_pairs)
            )

            denom = add_pair_mse[ii].mean()

            for name in boot:

                boot[name].append(
                    100 * (
                        1
                        - pair_losses[name][ii].mean()
                        / denom
                    )
                )


        # Calculate and format CIs
        for name in boot:

            ci_lower, ci_upper = np.quantile(
                boot[name],
                [0.025, 0.975]
            )

            df.loc[
                name,
                "MSE Improv. 95% CI"
            ] = (
                f"[{ci_lower:.2f}%, "
                f"{ci_upper:.2f}%]"
            )


        df.loc[
            "Additive Baseline",
            "MSE Improv. 95% CI"
        ] = "-"


        # 5. Calculate Pearson Correlation
        # Improvement (MLP vs Linear)

        mlp_pearson = df.loc[
            "MLP",
            "Pearson (Implied vs True Residual)"
        ]

        lin_pearson = df.loc[
            "Linear Regression",
            "Pearson (Implied vs True Residual)"
        ]


        if lin_pearson != 0:

            pearson_improvement = (
                (mlp_pearson - lin_pearson)
                / abs(lin_pearson)
            ) * 100

            df.loc[
                "MLP",
                "Pearson Improv. vs Linear (%)"
            ] = pearson_improvement


        df.loc[
            "Linear Regression",
            "Pearson Improv. vs Linear (%)"
        ] = 0.0


        # Use np.nan instead of a string
        # to keep the column numerically typed
        df.loc[
            "Additive Baseline",
            "Pearson Improv. vs Linear (%)"
        ] = np.nan


        # ========================================================
        # SAVE RESULTS FROM THIS NETWORK/TRAINING RUN
        # ========================================================

        run_df = df.reset_index()

        run_df.insert(
            0,
            "Training Seed",
            training_seed
        )

        run_df.insert(
            0,
            "Network Seed",
            network_seed
        )

        all_results.append(run_df)


        # Full results accumulated so far
        all_results_df = pd.concat(
            all_results,
            ignore_index=True
        )

        # Save after every training run so that completed
        # runs are not lost if a later run fails.
        all_results_df.to_csv(
            "../grn_training_results.csv",
            index=False
        )


        # Compact one-row-per-run RMSE table
        # MLP vs regression vs additive
        rmse_by_run = all_results_df.pivot(
            index=[
                "Network Seed",
                "Training Seed"
            ],
            columns="Model",
            values="RMSE"
        ).reset_index()

        rmse_by_run.columns.name = None

        rmse_by_run.to_csv(
            "../grn_training_rmse_by_run.csv",
            index=False
        )


        # Format the DataFrame for clean output
        display_cols = [
            "RMSE",
            "MSE Improvement vs Additive (%)",
            "MSE Improv. 95% CI",
            "Pearson (Implied vs True Residual)",
            "Pearson Improv. vs Linear (%)",
            "Fraction Pairs < Additive MSE"
        ]


        print(
            f"\nNetwork seed {network_seed}, "
            f"training seed {training_seed}"
        )

        display(
            df[
                display_cols
            ].round(4).fillna("-")
        )


# ================================================================
# FINAL RESULT TABLES
# ================================================================

# 10 networks x 10 training runs x 3 models = 300 rows
all_results_df = pd.concat(
    all_results,
    ignore_index=True
)

# 100 rows, one for each Network Seed / Training Seed combination
rmse_by_run = all_results_df.pivot(
    index=[
        "Network Seed",
        "Training Seed"
    ],
    columns="Model",
    values="RMSE"
).reset_index()

rmse_by_run.columns.name = None

display(all_results_df)
display(rmse_by_run)

Network 0: single-gene KOs:   0%|          | 0/256 [00:00<?, ?it/s]

[Parallel(n_jobs=64)]: Using backend LokyBackend with 64 concurrent workers.
[Parallel(n_jobs=64)]: Done   8 tasks      | elapsed:   37.8s
[Parallel(n_jobs=64)]: Done  21 tasks      | elapsed:   38.0s
[Parallel(n_jobs=64)]: Done  34 tasks      | elapsed:   38.1s
[Parallel(n_jobs=64)]: Done  49 tasks      | elapsed:   38.2s
[Parallel(n_jobs=64)]: Done  64 tasks      | elapsed:   38.4s
[Parallel(n_jobs=64)]: Done  81 tasks      | elapsed:   38.5s
[Parallel(n_jobs=64)]: Done  98 tasks      | elapsed:   38.6s
[Parallel(n_jobs=64)]: Done 117 tasks      | elapsed:   38.9s
[Parallel(n_jobs=64)]: Done 136 tasks      | elapsed:   39.1s
[Parallel(n_jobs=64)]: Done 157 tasks      | elapsed:   39.2s
[Parallel(n_jobs=64)]: Done 178 tasks      | elapsed:   39.4s
[Parallel(n_jobs=64)]: Done 201 tasks      | elapsed:   39.6s
[Parallel(n_jobs=64)]: Done 224 tasks      | elapsed:   39.8s
[Parallel(n_jobs=64)]: Done 249 tasks      | elapsed:   40.0s
[Parallel(n_jobs=64)]: Done 274 tasks      | elapsed:  

network 0, training 0, epoch 1: train scaled MSE 0.82848; val RMSE 0.01920971; lr 0.001; 1s
network 0, training 0, epoch 10: train scaled MSE 0.0427469; val RMSE 0.003601856; lr 0.001; 3s
network 0, training 0, epoch 20: train scaled MSE 0.0261598; val RMSE 0.002339018; lr 0.001; 5s
network 0, training 0, epoch 30: train scaled MSE 0.0290917; val RMSE 0.002720074; lr 0.001; 7s
network 0, training 0, epoch 40: train scaled MSE 0.00812519; val RMSE 0.002214951; lr 0.001; 9s
network 0, training 0, epoch 50: train scaled MSE 0.00508276; val RMSE 0.002078251; lr 0.001; 11s
network 0, training 0, epoch 60: train scaled MSE 0.00828798; val RMSE 0.002134715; lr 0.001; 13s
network 0, training 0, epoch 70: train scaled MSE 0.00350614; val RMSE 0.001922647; lr 0.001; 15s
network 0, training 0, epoch 80: train scaled MSE 0.00307815; val RMSE 0.001939174; lr 0.001; 17s
network 0, training 0, epoch 90: train scaled MSE 0.00194486; val RMSE 0.001834042; lr 0.0005; 19s
network 0, training 0, epoch 100

,RMSE,MSE Improvement vs Additive (%),MSE Improv. 95% CI,Pearson (Implied vs True Residual),Pearson Improv. vs Linear (%),Fraction Pairs < Additive MSE
Model,,,,,,
Additive Baseline,0.0022,0.0000,-,0.0000,-,0.0000
Linear Regression,0.0022,-2.7916,"[-9.41%, 0.60%]",0.0741,0.0,0.1165
MLP,0.0023,-8.2798,"[-24.85%, 3.28%]",0.1900,156.4282,0.0121


network 0, training 1, epoch 1: train scaled MSE 0.824262; val RMSE 0.01895431; lr 0.001; 0s
network 0, training 1, epoch 10: train scaled MSE 0.0222366; val RMSE 0.004400522; lr 0.001; 2s
network 0, training 1, epoch 20: train scaled MSE 0.00751253; val RMSE 0.003602203; lr 0.001; 4s
network 0, training 1, epoch 30: train scaled MSE 0.00539413; val RMSE 0.003129729; lr 0.001; 6s
network 0, training 1, epoch 40: train scaled MSE 0.00446851; val RMSE 0.003126967; lr 0.001; 9s
network 0, training 1, epoch 50: train scaled MSE 0.00505875; val RMSE 0.003062358; lr 0.001; 11s
network 0, training 1, epoch 60: train scaled MSE 0.00351823; val RMSE 0.002994588; lr 0.0005; 13s
network 0, training 1, epoch 70: train scaled MSE 0.00314423; val RMSE 0.002970315; lr 0.0005; 15s
network 0, training 1, epoch 80: train scaled MSE 0.00308342; val RMSE 0.003116925; lr 0.0005; 17s
network 0, training 1, epoch 90: train scaled MSE 0.00271094; val RMSE 0.002925393; lr 0.0005; 19s
network 0, training 1, epo

...

,RMSE,MSE Improvement vs Additive (%),MSE Improv. 95% CI,Pearson (Implied vs True Residual),Pearson Improv. vs Linear (%),Fraction Pairs < Additive MSE
Model,,,,,,
Additive Baseline,0.0055,-0.0000,-,0.0000,-,0.0000
Linear Regression,0.0055,0.8785,"[-7.93%, 5.04%]",0.1813,0.0,0.0221
MLP,0.0040,48.1078,"[34.02%, 58.84%]",0.6950,283.2296,0.0119


Network 9: single-gene KOs:   0%|          | 0/256 [00:00<?, ?it/s]

[Parallel(n_jobs=64)]: Using backend LokyBackend with 64 concurrent workers.
[Parallel(n_jobs=64)]: Done   8 tasks      | elapsed:   27.2s
[Parallel(n_jobs=64)]: Done  21 tasks      | elapsed:   27.2s
[Parallel(n_jobs=64)]: Done  34 tasks      | elapsed:   27.4s
[Parallel(n_jobs=64)]: Done  49 tasks      | elapsed:   27.5s
[Parallel(n_jobs=64)]: Done  64 tasks      | elapsed:   27.6s
[Parallel(n_jobs=64)]: Done  81 tasks      | elapsed:   27.9s
[Parallel(n_jobs=64)]: Done  98 tasks      | elapsed:   28.0s
[Parallel(n_jobs=64)]: Done 117 tasks      | elapsed:   28.1s
[Parallel(n_jobs=64)]: Done 136 tasks      | elapsed:   28.2s
[Parallel(n_jobs=64)]: Done 157 tasks      | elapsed:   28.5s
[Parallel(n_jobs=64)]: Done 178 tasks      | elapsed:   28.7s
[Parallel(n_jobs=64)]: Done 201 tasks      | elapsed:   28.8s
[Parallel(n_jobs=64)]: Done 224 tasks      | elapsed:   29.1s
[Parallel(n_jobs=64)]: Done 249 tasks      | elapsed:   29.3s
[Parallel(n_jobs=64)]: Done 274 tasks      | elapsed:  

network 9, training 0, epoch 1: train scaled MSE 0.802468; val RMSE 0.01628833; lr 0.001; 0s
network 9, training 0, epoch 10: train scaled MSE 0.028881; val RMSE 0.003269627; lr 0.001; 2s
network 9, training 0, epoch 20: train scaled MSE 0.00981673; val RMSE 0.001846144; lr 0.001; 4s
network 9, training 0, epoch 30: train scaled MSE 0.00887541; val RMSE 0.002136649; lr 0.001; 6s
network 9, training 0, epoch 40: train scaled MSE 0.00487722; val RMSE 0.001933818; lr 0.001; 8s
network 9, training 0, epoch 50: train scaled MSE 0.00389312; val RMSE 0.001472036; lr 0.001; 10s
network 9, training 0, epoch 60: train scaled MSE 0.00319794; val RMSE 0.001471669; lr 0.001; 12s
network 9, training 0, epoch 70: train scaled MSE 0.00220839; val RMSE 0.001456227; lr 0.0005; 14s
network 9, training 0, epoch 80: train scaled MSE 0.00286423; val RMSE 0.001630138; lr 0.0005; 16s
network 9, training 0, epoch 90: train scaled MSE 0.00177864; val RMSE 0.001332413; lr 0.0005; 18s
network 9, training 0, epoch

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [4]:
mlp_ci_table = (
    all_results_df
    .query("Model == 'MLP'")
    .pivot(
        index="Training Seed",
        columns="Network Seed",
        values="MSE Improv. 95% CI"
    )
)

display(mlp_ci_table)

Network Seed,0,1,2,3,4,5,6,7,8,9
Training Seed,,,,,,,,,,
0,"[-24.85%, 3.28%]","[-43.83%, -9.31%]","[-54.01%, -25.27%]","[-40.89%, -10.22%]","[-6.95%, 23.34%]","[25.58%, 50.16%]","[0.69%, 34.45%]","[-4.86%, 23.09%]","[10.91%, 51.03%]","[16.49%, 55.89%]"
1,"[-23.36%, 23.99%]","[-45.20%, -9.71%]","[-42.76%, -23.48%]","[-56.67%, -6.03%]","[6.20%, 26.03%]","[27.24%, 51.40%]","[12.48%, 32.52%]","[5.13%, 24.23%]","[-13.17%, 53.43%]","[10.08%, 49.75%]"
2,"[-14.63%, 9.26%]","[-14.96%, 4.80%]","[-48.85%, -21.75%]","[-57.96%, -16.20%]","[-6.43%, 15.82%]","[32.55%, 50.12%]","[12.93%, 37.45%]","[20.37%, 38.74%]","[-53.38%, 48.32%]","[21.45%, 48.53%]"
3,"[-84.41%, -27.44%]","[-15.31%, 11.93%]","[-38.47%, -14.23%]","[-64.66%, -24.72%]","[1.25%, 27.20%]","[27.62%, 48.32%]","[-6.06%, 25.25%]","[5.56%, 27.92%]","[34.99%, 51.18%]","[30.74%, 55.55%]"
4,"[-14.36%, 26.40%]","[-29.63%, 3.21%]","[-42.73%, -18.27%]","[-51.91%, -14.74%]","[0.91%, 24.48%]","[36.35%, 57.50%]","[4.57%, 37.25%]","[-4.19%, 24.95%]","[32.56%, 60.46%]","[11.05%, 55.84%]"
5,"[-57.69%, 0.82%]","[-28.90%, 3.57%]","[-42.45%, -16.05%]","[-44.32%, -10.32%]","[-0.49%, 23.94%]","[29.50%, 52.16%]","[8.56%, 28.80%]","[12.57%, 34.03%]","[23.24%, 67.47%]","[30.24%, 56.49%]"
6,"[-22.65%, 0.01%]","[-23.08%, 13.60%]","[-64.65%, -34.68%]","[-54.16%, -17.67%]","[-1.43%, 23.71%]","[24.41%, 47.68%]","[1.42%, 31.80%]","[14.59%, 35.01%]","[31.83%, 59.18%]","[21.74%, 53.11%]"
7,"[-29.59%, 20.40%]","[-15.65%, 28.78%]","[-52.45%, -25.50%]","[-51.47%, -8.72%]","[1.29%, 28.33%]","[28.83%, 46.98%]","[10.94%, 41.30%]","[18.83%, 35.14%]","[29.83%, 71.65%]","[28.69%, 59.16%]"
8,"[-11.19%, 32.98%]","[-11.62%, 23.04%]","[-49.88%, -20.89%]","[-50.64%, -21.71%]","[-12.19%, 17.40%]","[25.55%, 50.71%]","[11.79%, 36.00%]","[15.70%, 35.26%]","[11.98%, 53.73%]","[31.96%, 46.31%]"


In [5]:
mlp_ci_table = (
    all_results_df
    .query("Model == 'MLP'")
    .pivot(
        index="Training Seed",
        columns="Network Seed",
        values="MSE Improvement vs Additive (%)"
    )
)

display(mlp_ci_table)

Network Seed,0,1,2,3,4,5,6,7,8,9
Training Seed,,,,,,,,,,
0,-8.279761,-25.080205,-37.420500,-22.489317,14.265073,38.916124,20.912626,9.605272,34.088205,39.189121
1,13.467215,-25.743465,-31.585585,-26.609285,17.945392,39.628655,23.783974,15.730771,29.350265,32.828715
2,0.684769,-3.929866,-33.624948,-31.373398,7.047897,41.687451,25.794249,29.816249,5.531624,36.342548
3,-46.188922,-0.214643,-24.129350,-39.461119,17.944355,38.718429,13.427269,17.493844,43.588937,45.185535
4,5.757635,-12.105118,-29.022836,-29.313645,15.620344,47.585345,22.365626,11.777680,48.177324,36.716998
5,-21.548852,-11.153580,-26.189068,-23.366090,15.151538,41.795919,20.311578,24.249073,50.922569,44.913896
6,-8.533990,-4.323529,-46.454928,-32.723388,13.766134,36.624880,19.377537,25.678687,46.676424,40.529287
7,5.629139,7.495240,-37.544963,-25.063104,18.752930,38.616331,27.984417,27.512118,56.194618,45.168782
8,18.908670,6.548808,-33.086014,-33.714648,6.701831,37.909628,25.307224,26.344443,36.106447,39.374934
